# Mass & Power Margin Analysis

**Mission:** Phobos Sample Return (PhSR)

This notebook computes mass and power margins for the spacecraft, comparing
the current mass equipment list and power budget against system capacities.

---

## 1. Mass Margins

The mass budget tracks Current Best Estimate (CBE) masses for each subsystem,
applies component-level margins to obtain the Maximum Expected Value (MEV),
then adds a 25% system-level margin on the dry mass. We compare the total wet
mass against the ESPA-Grande launch vehicle capacity.

In [ ]:
# Mass budget summary values from the mass equipment list
dry_mass_cbe    = 537.45   # kg — CBE dry mass total
dry_mass_mev    = 590.64   # kg — MEV dry mass (component margins)
system_margin   = 147.66   # kg — 25% system margin
dry_mass_margin = 738.3   # kg — dry mass with margin
propellant_mass = 1117.7   # kg — total propellant
wet_mass        = 1856.0   # kg — total wet mass

println("Dry Mass CBE:          $(dry_mass_cbe) kg")
println("Dry Mass MEV:          $(dry_mass_mev) kg")
println("System Margin (25%):   $(system_margin) kg")
println("Dry Mass with Margin:  $(dry_mass_margin) kg")
println("Propellant Mass:       $(propellant_mass) kg")
println("Total Wet Mass:        $(wet_mass) kg")

In [ ]:
# Launch vehicle capacity (ESPA-Grande class)
lv_capacity = 2200.0  # kg

mass_margin_kg      = lv_capacity - wet_mass
mass_margin_percent = (mass_margin_kg / lv_capacity) * 100.0

println("Launch Vehicle Capacity:  $(lv_capacity) kg")
println("Wet Mass:                $(wet_mass) kg")
println("Mass Margin:             $(round(mass_margin_kg, digits=1)) kg")
println("Mass Margin:             $(round(mass_margin_percent, digits=1)) %")

if mass_margin_percent >= 0
    println("\nSTATUS: POSITIVE MARGIN — spacecraft fits within LV capacity.")
else
    println("\nSTATUS: NEGATIVE MARGIN — mass reduction required!")
end

### Mass Budget Breakdown

| Item | Value (kg) |
|:--|--:|
| Dry Mass CBE | 537.45 |
| Dry Mass MEV (component margins) | 590.64 |
| System Margin (25%) | 147.66 |
| **Dry Mass with Margin** | **738.3** |
| Bipropellant | 1060.8 |
| Monopropellant | 56.9 |
| **Total Wet Mass** | **1856.0** |

---

## 2. Power Margins

The power budget defines five operational modes. For each mode, we compare
the total power draw against the solar array power available at Mars end-of-life
(1503 W) to determine the power margin.

In [ ]:
# Solar array power available at Mars EOL
# The value from the solar array spec is a quantity string; we extract the number.
solar_power_eol = parse(Float64, split("1503 W")[1])

println("Solar Array Power (Mars EOL): $(solar_power_eol) W")

In [ ]:
# Power draw per mode (from power budget totals row)
mode_names = ["Cruise", "Science Mapping", "Communications", "TAG Maneuver", "Safe Mode"]
mode_power = [
    167,   # Cruise
    242,   # Science Mapping
    215,   # Communications
    339,   # TAG Maneuver
    167    # Safe Mode
]

println("Mode Power Draws:")
for (name, power) in zip(mode_names, mode_power)
    println("  $(rpad(name, 20)) $(power) W")
end

In [ ]:
# Compute power margins for each mode
println("Power Margin Analysis (vs Solar Array Mars EOL = $(solar_power_eol) W)")
println("="^70)
println(rpad("Mode", 22), rpad("Draw (W)", 12), rpad("Margin (W)", 14), "Margin (%)")
println("-"^70)

for (name, power) in zip(mode_names, mode_power)
    margin_w = solar_power_eol - power
    margin_pct = (margin_w / solar_power_eol) * 100.0
    status = margin_pct >= 30 ? " [OK]" : margin_pct >= 0 ? " [LOW]" : " [OVER]" 
    println(rpad(name, 22), rpad(string(power), 12), rpad(string(round(margin_w, digits=1)), 14), "$(round(margin_pct, digits=1))%$(status)")
end

### Power Mode Descriptions

| Mode | Description | Total Power |
|:--|:--|--:|
| Cruise | Quiescent interplanetary transit; C&DH active, ADCS in wheel-control, SDST receive-only, survival heaters | 167 W |
| Science Mapping | Phobos observation from QSO; camera and GRNS active, HGA downlink, full ADCS | 242 W |
| Communications | High-rate data downlink via HGA to DSN; TWTA at full power, no science instruments | 215 W |
| TAG Maneuver | Touch-and-go descent and sampling at Phobos surface; reaction wheels at peak, RCS active, sample system powered | 339 W |
| Safe Mode | Minimum-power survival mode; sun-pointing via sun sensors and RCS, LGA downlink, maximum heaters | 167 W |

---

## 3. Battery Sizing Check

During Mars eclipse, the battery must sustain reduced operations.
We verify the battery capacity supports the worst-case eclipse scenario.

In [ ]:
# Battery parameters
battery_capacity = parse(Float64, split("2400 W*hr")[1])
dod_limit_str    = "40 %"
dod_limit        = parse(Float64, split(dod_limit_str)[1]) / 100.0

# Eclipse parameters
eclipse_duration_str = "75 min"
eclipse_duration_hr  = parse(Float64, split(eclipse_duration_str)[1]) / 60.0  # convert min to hr

eclipse_power_str = "600 W"
eclipse_power     = parse(Float64, split(eclipse_power_str)[1])

# Usable energy at DOD limit
usable_energy    = battery_capacity * dod_limit
required_energy  = eclipse_power * eclipse_duration_hr
energy_margin    = usable_energy - required_energy
energy_margin_pct = (energy_margin / usable_energy) * 100.0

println("Battery Capacity:      $(battery_capacity) Wh")
println("DOD Limit:             $(dod_limit * 100)%")
println("Usable Energy:         $(usable_energy) Wh")
println("Eclipse Duration:      $(round(eclipse_duration_hr, digits=2)) hr")
println("Eclipse Power Load:    $(eclipse_power) W")
println("Required Energy:       $(required_energy) Wh")
println("Energy Margin:         $(round(energy_margin, digits=1)) Wh")
println("Energy Margin:         $(round(energy_margin_pct, digits=1))%")

---

## Summary

This analysis shows the mass and power margins for the spacecraft at the
current design point. All margins should be reviewed against project margin
policy (typically >30% power margin and >15% mass margin at PDR).

---

*Report generated by VVERDAD*